In [2]:
import os
import glob
from google import genai
import re
import shutil
import unicodedata
import time
from dotenv import load_dotenv
from xai_sdk import Client
from xai_sdk.chat import user, system
from langchain_text_splitters import RecursiveCharacterTextSplitter, TextSplitter, SpacyTextSplitter
import pdfplumber
from google import genai
from google.genai import types
import json

In [10]:
MODEL = "grok-4.20-0309-reasoning"
book_ratio = 50 #What % of the book has to be left

In [11]:
def get_paths():
    pdf_list = glob.glob("input/*.pdf")
    pdf_list = [re.sub(r'\\', '/', pdf) for pdf in pdf_list]
    print(pdf_list[0])
    #re.replace("\", "/")

In [12]:
def extract_text(pdf_path):
    """Extract all text from a PDF. Returns (full_text, page_count)."""
    pages = []
    with pdfplumber.open(pdf_path) as pdf:
        page_count = len(pdf.pages)
        for page in pdf.pages:
            text = page.extract_text()
            if text:
                pages.append(text)
    return "\n\n".join(pages), page_count

In [13]:
pdf_list = glob.glob("input/*.pdf")
pdf = pdf_list[1]
text, pageCount = extract_text(pdf)
print(text, pageCount)

Jonas Biliūnas
LIŪDNA PASAKA
BALTASAI ŠEŠĖLIS
Prakalbos vietoje
O jaunosios dienos mano! Kaip melsvam ore gervės nykstat jūs, tiktai ką pasirodžiusios... Žiūriu į jus
pralėkusias, kaip į sapną gražų, ir matau tik, kad jau artinas ruduo gyvenimo mano.
O Ramūta! Kodėl nepažinau tavęs, dar mažutis, nekaltas būdamas! Tada tokios gražios, laimingos
buvo dienos. Būtume, už rankų susitvėrę, Šventosios pakrančiais vaikščioję, lakštingalos giesmių klausę.
Būčiau tave po miškus ir krūmus išvedžiojęs, paukščių lizdus parodęs, išsirpusių uogų parinkęs. Bet tu
buvai toli nuo manęs, toli, ir aš net nežinojau, kad tu gyveni pasaulyje...
Taip, nežinojau... Bet jaučiau... Nuo pat mažų dienų tavo paveikslas mano širdyje gyveno. O kai,
mokiniu būdamas, traukiniu gimnazijon važiuodavau ir tavo tėvynės mirguojančius klonius ir laukus ir
gimtąjį sodžių pro langą išvysdavau, mano krūtinėje netikėtai sujudėdavo naujas, geras jautimas: kaip
paukštis pro šalį pralėkdavau, bet ilgai dar į išsitiesusį paveiksląži

In [ ]:
def chunk_book(text):
    chunker = RecursiveCharacterTextSplitter(chunk_size=11500, chunk_overlap=200)
    chunks = chunker.split_text(text)
    print(f"Chunks: {len(chunks)}")

    return chunks


In [28]:
chunks = chunk_book()

Chunks: 5


In [17]:
print(chunks[0])
print(f"\n\n\n\n\n{chunks[1]}")
print(f"\n\n\n\n\n{chunks[2]}")

Jonas Biliūnas
LIŪDNA PASAKA
BALTASAI ŠEŠĖLIS
Prakalbos vietoje
O jaunosios dienos mano! Kaip melsvam ore gervės nykstat jūs, tiktai ką pasirodžiusios... Žiūriu į jus
pralėkusias, kaip į sapną gražų, ir matau tik, kad jau artinas ruduo gyvenimo mano.
O Ramūta! Kodėl nepažinau tavęs, dar mažutis, nekaltas būdamas! Tada tokios gražios, laimingos
buvo dienos. Būtume, už rankų susitvėrę, Šventosios pakrančiais vaikščioję, lakštingalos giesmių klausę.
Būčiau tave po miškus ir krūmus išvedžiojęs, paukščių lizdus parodęs, išsirpusių uogų parinkęs. Bet tu
buvai toli nuo manęs, toli, ir aš net nežinojau, kad tu gyveni pasaulyje...
Taip, nežinojau... Bet jaučiau... Nuo pat mažų dienų tavo paveikslas mano širdyje gyveno. O kai,
mokiniu būdamas, traukiniu gimnazijon važiuodavau ir tavo tėvynės mirguojančius klonius ir laukus ir
gimtąjį sodžių pro langą išvysdavau, mano krūtinėje netikėtai sujudėdavo naujas, geras jautimas: kaip
paukštis pro šalį pralėkdavau, bet ilgai dar į išsitiesusį paveiksląži

In [ ]:
def convert_pdfs_to_ascii():
    pdf_list = glob.glob("input/*.pdf")
    pdf_list = [re.sub(r'\\', '/', pdf) for pdf in pdf_list]

    for pdf in pdf_list:
        mod_pdf = pdf.replace("input", "ascii_input")
        if os.path.exists(mod_pdf):
            print(f"{mod_pdf} already exists")
        else:
            base = os.path.basename(pdf)
            ascii_base = unicodedata.normalize("NFKD", base).encode("ascii", "ignore").decode("ascii")
            ascii_base = re.sub(r"[^A-Za-z0-9._-]+", "_", ascii_base)
            safe_path = os.path.join("ascii_input", ascii_base)
            # Keep original file; just make a copy with safe name
            shutil.copy2(pdf, safe_path)
            print(f"{mod_pdf} created")
    return None

ascii_input/Francas_Kafka._Metamorfozė.LHV900.pdf created
ascii_input/Jonas_Biliūnas._tik_Liūdna_pasaka.LG1800.pdf created


In [ ]:
def upload_pdfs_to_gemini():
    client = genai.Client()
    pdfs=glob.glob("ascii_input/*.pdf")
    file_pdf_list = []

    # Upload each PDF
    for pdf in pdfs:
        myfile = client.files.upload(file=pdf)
        file_name = myfile.name
        myfile = client.files.get(name=file_name)
        print(myfile)
        file_pdf_list.append({"book_name": os.path.basename(pdf), "file": myfile})

        # Wait for the file to be active
        while myfile.status != "ACTIVE":
            time.sleep(1)
            myfile = client.files.get(name=file_name)
            print(myfile)

    return file_pdf_list

In [ ]:
def upload_pdfs_to_grok():
    pdfs=glob.glob("ascii_input/*.pdf")
    grok_file_pdf_list = []

    # Upload each PDF
    load_dotenv()
    client = Client(api_key=os.getenv("XAI_API_KEY"))
    for pdf in pdfs:
        file = client.files.upload(pdf, expires_after=82800) #expires after 23 hours
        grok_file_pdf_list.append({"book_name": os.path.basename(pdf), "file": file})



    return grok_file_pdf_list

In [ ]:
GEMINI_BATCH_MODEL = "gemini-2.5-flash-preview-04-17"

def call_gemini_batch_old(chunks, gemini_file, ratio_pct=book_ratio):
    ratio_pct = book_ratio

    inline_requests = []

    for i, chunk_text in enumerate(chunks):
        chunk_num = i + 1
        target_len = int(len(chunk_text) * ratio_pct / 100)
        lo_len = int(target_len * 0.90)
        hi_len = int(target_len * 1.1)




        system_text = GEMINI_SYSTEM.format(ratio_pct=ratio_pct)
        user_text = GEMINI_USER.format(
            ratio_pct=ratio_pct,
            chunk_num=chunk_num,
            target_len=target_len,
            lo_len=lo_len,
            hi_len=hi_len,
            chunk_start=chunk_text[:500],
            chunk_end=chunk_text[-500:]
        )

        inline_requests.append({
            'system_instruction': {'parts': [{'text': system_text}]},
            'contents': [{
                'role': 'user',
                'parts': [
                    {'file_data': {'file_uri': gemini_file.uri, 'mime_type': 'application/pdf'}},
                    {'text': user_text},
                ],
            }],
        })

    gemini_client = genai.Client()
    batch_job = gemini_client.batches.create(
        model=GEMINI_BATCH_MODEL,
        src=inline_requests,
        config={'display_name': 'book-compression'},
    )
    print(f"Batch created: {batch_job.name}  state: {batch_job.state}")

    while True:
        state = str(batch_job.state)
        if 'SUCCEEDED' in state:
            break
        if 'FAILED' in state or 'CANCELLED' in state or 'EXPIRED' in state:
            raise RuntimeError(f"Batch job ended with state: {state}")
        time.sleep(30)
        batch_job = gemini_client.batches.get(name=batch_job.name)
        print(f"  state: {batch_job.state}")

    return [
        resp.response.candidates[0].content.parts[0].text
        for resp in batch_job.dest.inlined_responses
    ]

In [ ]:


def call_gemini_batch(chunks, gemini_file, ratio_pct=book_ratio):
    # 1. Initialize Client
    load_dotenv()
    client = genai.Client()
    
    file = gemini_file["file"]
    fileID = file.id
    book_name = gemini_file["book_name"]
    # Ensure ratio_pct is handled if not passed (using your variable name)
    if ratio_pct is None:
        ratio_pct = book_ratio 

    batch_filename = "gemini_batch_input.jsonl"
    
    # 2. Generate the JSONL file
    with open(batch_filename, 'w', encoding='utf-8') as f:
        for i, chunk_text in enumerate(chunks):
            chunk_num = i + 1
            target_len = int(len(chunk_text) * ratio_pct / 100)
            
            # Setup prompt variables
            system_text = GEMINI_SYSTEM.format(ratio_pct=ratio_pct)
            user_text = GEMINI_USER.format(
                ratio_pct=ratio_pct,
                chunk_num=chunk_num,
                target_len=target_len,
                lo_len=int(target_len * 0.90),
                hi_len=int(target_len * 1.1),
                chunk_start=chunk_text[:500],
                chunk_end=chunk_text[-500:],
                file_id = fileID
            )

            # Construct the specific JSONL structure
            # The 'key' allows you to identify the result even if it returns out of order
            request_data = {
                "key": f"{book_name}_{chunk_num:03d}", 
                "request": {
                    "method": "generateContent", # Required in the 2026 JSONL schema
                    "model": f"models/{GEMINI_BATCH_MODEL}",
                    "system_instruction": {"parts": [{"text": system_text}]},
                    "contents": [{
                        "role": "user",
                        "parts": [
                            # Reference the file URI for 'Perfect Vision'
                            {"file_data": {"file_uri": file.uri, "mime_type": "application/pdf"}},
                            {"text": user_text}
                        ]
                    }]
                }
            }
            f.write(json.dumps(request_data) + '\n')

    # 3. Upload the JSONL file to Gemini
    print(f"Uploading batch instructions...")
    batch_input_file = client.files.upload(path=batch_filename)

    # 4. Create the Batch Job
    batch_job = client.batches.create(
        model=GEMINI_BATCH_MODEL,
        src=batch_input_file.name, # Use the file name (files/xyz) as source
        config={'display_name': f'book-compression-{chunk_num}-chunks'}
    )

    print(f"✅ Batch Job Created!")
    print(f"Job Name: {batch_job.name}")
    print(f"Current State: {batch_job.state}")
    
    # Clean up local file
    os.remove(batch_filename)
    
    return batch_job

In [4]:
load_dotenv()
client = Client(api_key=os.getenv("XAI_API_KEY"))

In [ ]:
GROK_SYSTEM = """\
Tu esi lietuvių literatūros teksto atkūrimo redaktorius.

<constraints>
MUST: Naudok tik originalaus failo ir Gemini versijos turinį.
MUST: Galutinio teksto ilgis — ±10% Gemini versijos ilgio.
MUST: Išlaik autoriaus sakinių ritmą, leksiką ir pasakojimo toną.
MUST: Atkurk viską, kas prarasta — vardus, vietas, datas, dialogo fragmentus, emocines reakcijas, aplinkos detales.
NEVER: Nepridėk informacijos, kurios nėra originale.
NEVER: Grąžink tik tekstą — jokių komentarų, įžangų, paaiškinimų.
NEVER: Nerašyk AI stiliumi — tekstas turi skambėti kaip autorius.
</constraints>

<task>
Vienas praėjimas: surask, ko trūksta Gemini versijoje lyginant su originalu, įterpk į Gemini struktūrą, ištaisyk nenatūralią lietuvių kalbą. Jei reikia jungties su ankstesniu fragmentu — maksimaliai 1–2 frazės iš originalo turinio. Grąžink tik rezultatą.
</task>

<reasoning_discipline>
NEVER: Necituok ir nerašyk pilno teksto reasoning metu.
MUST: Reasoning — tik trūkstamų elementų sąrašas trumpais įrašais (pvz.: "vardas X → įterpti 3 par.", "dialogas → atkurti"). Jokio teksto perrašymo.
MUST: Kuo mažiau reasoning — kuo daugiau tiesiogiai į rezultatą.
</reasoning_discipline>
"""

GROK_USER = """\
<original_file.id>
{file}
</original_file.id>

<gemini_compressed_version>
{compressed}
</gemini_compressed_version>

Atkurk ir patobulink sutrumpintą fragmentą remdamasis originalu. Pateik tik galutinį tekstą.
Galutinis tekstas:\
"""

In [ ]:
GEMINI_SYSTEM = """\
Tu esi aukštos kvalifikacijos lietuvių literatūros redaktorius ir teksto trumpintojas.
Tavo užduotis yra sutrumpinti pateiktą knygos fragmentą TIKSLIAI iki {ratio_pct}% jo originalaus simbolių skaičiaus.

SVARBIOS TAISYKLĖS:
1. ILGIO REIKALAVIMAS — PRIVALOMAS: rezultato tekstas turi būti {ratio_pct}% originalaus fragmento ilgio (±5%).
2. Išlaik VISUS esminius siužeto įvykius, pagrindinių veikėjų vystymąsi ir svarbias scenas.
3. Išlaik autoriaus kalbos stilių ir toną kiek įmanoma.
4. Sumažink antrinius aprašymus, pasikartojančias mintis ir per ilgus dialogus.
5. Trumpink, bet NEKURK naujų faktų ar įvykių — tik rinktinai šalink.
6. Rezultatas turi būti sklandžiai skaitomas lietuviškas tekstas.
7. NEANALIZUOK ir NESKAIČIUOK — tiesiog pateik sutrumpintą tekstą be jokių komentarų, skaičiavimų ar įžangų.
8. Nepridėk jokių antraščių ar paaiškinimų — pradėk tiesiogiai nuo teksto.\
"""

GEMINI_USER = f"""\
Sutrumpink toliau pateiktą knygos fragmentą iki {ratio_pct}% jo dydžio. Tai {chunk_num} fragmentas.

fragmento pradžia:
{chunk_start}

fragmento pabaiga:
{chunk_end}

Tikslinis ilgis: {target_len} simbolių (leistinas diapazonas: {lo_len}–{hi_len}).

Sutrumpink TIK šį fragmentą — pilnas knygos PDF pridėtas kaip kontekstas, kad geriau suprastum siužetą, bet nereikia trumpinti visos knygos. Tik nuo nurodytos fragmento pradžios iki nurodytos fragmento pabaigos.

--- ORGINALAUS FAILO ID (file.id) ---
{file_id}
--- ORGINALAUS FAILO ID (file.id) PABAIGA ---

Pateik tik sutrumpintą fragmento tekstą. Jokių komentarų ar skaičiavimų.
Galutinis tekstas:\
"""

In [ ]:
def make_grok_prompt(file_id, gemini_chunk):
    user_prompt = user(GROK_USER.format(file=file_id, compressed=gemini_chunk))
    system_prompt = system(GROK_SYSTEM)

    return system_prompt, user_prompt

In [ ]:
def make_batch(model, gemini_chunk, file_id, batch_id):
    system_prompt, user_prompt = make_grok_prompt(file_id, gemini_chunk)

    chat = client.chat.create(
    model=model,
    batch_request_id=batch_id
)
    chat.append(system_prompt)
    chat.append(user_prompt)

    return chat

In [ ]:
def make_request(book_name, file_id, gemini_chunks):
    batch_requests = []

    batch = client.batch.create(batch_name=book_name)
    batch_id = batch.batch_id
    gemini_chunks = gemini_chunks

    for chunk in gemini_chunks:
        chunkBatch = make_batch(MODEL, chunk, file_id, batch_id)
        batch_requests.append(chunkBatch)
    
    client.batch.add(batch_id=batch_id, batch_requests=batch_requests)

    return batch_id

In [ ]:
def convert_books_gemini():
    batch_list = []

    geminiFiles = upload_pdfs_to_gemini()
    howManyBatches = len(geminiFiles)
    text, pageCount = extract_text(pdf)
    chunks = chunk_book(text)

    #Gemini shortening
    for file in geminiFiles:
        batch_list.append(call_gemini_batch(chunks, file))

    return batch_list

In [ ]:
def wait_for_gemini_batch(batchList):
    load_dotenv()
    client = genai.Client()

    batches = []
    is_proccesed = [False]*len(batchList)

    while not all(is_proccesed):
        for i, batch in enumerate(batchList):
            job_status = client.batches.get(name=batch.name)
            if job_status.state == 'SUCCEEDED' and is_proccesed[i] == False:
                print(f"Results are ready at: {job_status.output_file_id}")
                content = client.files.download(name=job_status.output_file_id)
        
        
                results = []
                # The output is a JSONL where each line is a result for one chunk
                for line in content.decode().splitlines():
                    if not line.strip(): continue
                    
                    data = json.loads(line)
                    chunk_key = data.get("key") # e.g., "chunk_001"
                    
                    # Extract the actual text generated by Grok/Gemini
                    try:
                        generated_text = data['response']['candidates'][0]['content']['parts'][0]['text']
                        results.append({
                            "key": chunk_key,
                            "text": generated_text
                        })
                    except (KeyError, IndexError):
                        print(f"Warning: Could not parse response for {chunk_key}")

                # Sort results by key to make sure the book is in the right order
                batches.append({"name": chunk_key, "content": results.sort(key=lambda x: x['key'])})

                #Mark as downloaded
                is_proccesed[i] = True


        time.sleep(60)
    return batches

In [ ]:
def convert_books_grok():
    grokFiles = upload_pdfs_to_grok()
    text, pageCount = extract_text(pdf)
    gemini_chunks = wait_for_gemini_batch()

    batchIds = []

    for i, file in enumerate(grokFiles):
        bookName = file["book_name"]
        file_id = file["file"].id
        batchIds.append(make_request(bookName, file_id, gemini_chunks[i]["content"]))

    return batchIds

In [ ]:
def wait_for_grok_batch(batchID):
    load_dotenv()
    client = Client(api_key=os.getenv("XAI_API_KEY"))
    # Paginate through all results
    all_succeeded = []
    all_failed = []
    pagination_token = None
    while True:
        # Fetch a page of results (limit controls page size)
        page = client.batch.list_batch_results(
            batch_id=batchID,
            limit=100,
            pagination_token=pagination_token,
        )
        
        # Collect results from this page
        all_succeeded.extend(page.succeeded)
        all_failed.extend(page.failed)
        
        # Check if there are more pages
        if page.pagination_token is None:
            break
        pagination_token = page.pagination_token
    # Process results - handle different response types
    print(f"Successfully processed: {len(all_succeeded)} requests")
    for result in all_succeeded:
        rid = result.batch_request_id
        resp = result.proto.response
        if resp.HasField("completion_response"):
            # Chat completion response
            print(f"[{rid}] {result.response.content}")
            print(f"  Tokens used: {result.response.usage.total_tokens}")
    if all_failed:
        print(f"\nFailed: {len(all_failed)} requests")
        for result in all_failed:
            print(f"[{result.batch_request_id}] Error: {result.error_message}")

    return all_succeeded, all_failed

SyntaxError: incomplete input (3337125347.py, line 1)